In [1]:
import pandas as pd

df = pd.read_pickle('/content/drive/MyDrive/EE244 PROJECT/sample_df_clean.pkl')


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd


class ChessFinalBoardDatasetWithELO(Dataset):
    def __init__(self, dataframe):
        self.df = dataframe

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        game_tensor = self.df.iloc[idx]['game_tensor']
        label = int(self.df.iloc[idx]['result'])
        final_board = game_tensor[-1]

        white_elo = self.df.iloc[idx]['white_elo']
        black_elo = self.df.iloc[idx]['black_elo']
        elo_diff = torch.tensor([(white_elo - black_elo) / 1000.0], dtype = torch.float32)

        return final_board.float(), elo_diff, torch.tensor(label, dtype = torch.long)

In [3]:
class ChessCNNWithELO(nn.Module):
    def __init__(self):
        super(ChessCNNWithELO, self).__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(12, 32, kernel_size = 3, padding = 1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size = 3, padding = 1),
            nn.ReLU(),
            nn.Flatten()
        )
        self.mlp = nn.Sequential(
            nn.Linear(64 * 8 * 8 + 1, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 2)
        )

    def forward(self, board, elo_diff):
        features = self.cnn(board)
        combined = torch.cat([features, elo_diff], dim = 1)
        return self.mlp(combined)

class ChessCNNRNNModel(nn.Module):
    def __init__(self):
        super(ChessCNNRNNModel, self).__init__()

        self.cnn = nn.Sequential(
            nn.Conv2d(12, 32, kernel_size = 3, padding = 1),
            nn.GELU(),
            nn.Conv2d(32, 64, kernel_size = 3, padding = 1),
            nn.GELU(),
            nn.Flatten()
        )

        self.cnn_output_size = 64 * 8 * 8
        self.rnn_input_size = self.cnn_output_size + 1

        self.rnn = nn.GRU(input_size = self.rnn_input_size, hidden_size = 256, batch_first = True)

        self.mlp = nn.Sequential(
            nn.Linear(256, 128),
            nn.GELU(),
            nn.Linear(128, 2)
        )

    def forward(self, board_tensor, evals):
        """
        board_tensor: (B, T, 12, 8, 8)
        evals: (B, T, 1)
        """
        B, T, C, H, W = board_tensor.shape
        boards_reshaped = board_tensor.view(B * T, C, H, W)
        cnn_features = self.cnn(boards_reshaped).view(B, T, -1)
        combined_features = torch.cat([cnn_features, evals], dim = -1)
        rnn_out, _ = self.rnn(combined_features)
        final_feature = rnn_out[:, -1, :]
        return self.mlp(final_feature)

from torch.utils.data import Dataset
import torch

class ChessGameDataset(Dataset):
    def __init__(self, dataframe):
        self.df = dataframe

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        game_tensor = self.df.iloc[idx]['game_tensor']
        eval_tensor = self.df.iloc[idx]['eval_tensor']
        label = int(self.df.iloc[idx]['result'])


        if eval_tensor.ndim == 1:
            eval_tensor = eval_tensor.unsqueeze(-1)

        return game_tensor.float(), eval_tensor.float(), torch.tensor(label, dtype = torch.long)


from torch.nn.utils.rnn import pad_sequence

def pad_collate(batch):
    """
    Pads game_tensor and eval_tensor along the time dimension.
    """
    game_tensors, eval_tensors, labels = zip(*batch)


    padded_game = pad_sequence(game_tensors, batch_first = True)
    padded_eval = pad_sequence(eval_tensors, batch_first = True)

    labels = torch.stack(labels)
    return padded_game, padded_eval, labels


In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dataset_test_2 = ChessGameDataset(df)
test_loader = DataLoader(dataset_test_2, batch_size = 1, shuffle = True, collate_fn = pad_collate)

all_eval_tensors = []

for _, eval_tensor, _ in test_loader:
    all_eval_tensors.append(eval_tensor)

all_eval_tensors = torch.cat(all_eval_tensors, dim = 0)

eval_mean_test = all_eval_tensors.mean()
eval_std_test = all_eval_tensors.std()
eval_mean_test = eval_mean_test.to(device)
eval_std_test = eval_std_test.to(device)




#load model in from path
#device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = torch.load('/content/drive/MyDrive/EE244 PROJECT/model_at_58.14.pt', weights_only = False).to(device)
model.eval()
#device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ChessCNNRNNModel(
  (cnn): Sequential(
    (0): Conv2d(12, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): GELU(approximate='none')
    (2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): GELU(approximate='none')
    (4): Flatten(start_dim=1, end_dim=-1)
  )
  (rnn): GRU(4097, 256, batch_first=True)
  (mlp): Sequential(
    (0): Linear(in_features=256, out_features=128, bias=True)
    (1): GELU(approximate='none')
    (2): Linear(in_features=128, out_features=2, bias=True)
  )
)

In [7]:
for boards, elo_diffs, labels in test_loader:
    boards, elo_diffs, labels = boards.to(device), elo_diffs.to(device), labels.to(device)
    with torch.no_grad():
        for board_tensor, eval_tensor, labels in test_loader:
            board_tensor = board_tensor.to(device)
            eval_tensor = eval_tensor.to(device)
            eval_tensor = (eval_tensor - eval_mean_test) / eval_std_test
            labels = labels.to(device)
            outputs = model(board_tensor, eval_tensor)
            _, predicted = torch.max(outputs, dim=1)
    break



labels = labels.cpu()[0]
predicted = predicted.cpu()[0]

print("Model that was at 58 percent accuracy\n")
if labels == 0:
    print("Ground truth win: Black")
else:
    print("Ground truth win: White")

if predicted == 0:
    print("Model predicted: Black")
else:
    print("Model predicted: White")


Model that was at 58 percent accuracy

Ground truth win: Black
Model predicted: Black


In [8]:
model = torch.load('/content/drive/MyDrive/EE244 PROJECT/model_at_64.15.pt', weights_only = False).to(device)
model.eval()
dataset_test = ChessFinalBoardDatasetWithELO(df)
test_loader = DataLoader(dataset_test, batch_size = 1, shuffle=False, num_workers = 0)


In [9]:
with torch.no_grad():
    for boards, elo_diffs, labels in test_loader:
        boards = boards.to(device)
        elo_diffs = elo_diffs.to(device)
        labels = labels.to(device)
        outputs = model(boards, elo_diffs)
        _, predicted = torch.max(outputs, dim=1)

labels = labels.cpu()[0]
predicted = predicted.cpu()[0]

print("Model that was at 64 percent accuracy\n")
if labels == 0:
    print("Ground truth win: Black")
else:
    print("Ground truth win: White")

if predicted == 0:
    print("Model predicted: Black")
else:
    print("Model predicted: White")

Model that was at 64 percent accuracy

Ground truth win: Black
Model predicted: Black
